# Music Annotation Project V8.8A
## Notebook 52: M5_REMOVE Policy Freeze and Replay

**Runtime:** CPU. In Google Colab select **Runtime > Run all**.

This notebook converts the latest complete Notebook 50 run into a durable V8.8A policy record. It does **not** retrain, recalibrate, move thresholds, reintroduce SAFE_PROTECT, or alter any saved Notebook 50 decision.

### Authoritative source rule
Unless `NOTEBOOK50_RUN_OVERRIDE` is set, the authoritative source is the **most recently modified complete Notebook 50 run directory** under the configured Notebook 50 roots. Completeness is determined from required review/control files, not directory name alone. If the newest candidate is incomplete, the notebook stops and reports it rather than silently selecting an older run.

### Fixed interpretation
- The deployable challenger is **M5_REMOVE = M3 minus SAFE_REMOVE**.
- SAFE_PROTECT is recorded but excluded from the runtime policy.
- Unsupported, uncertain, invalid or failed locations retain exact M3 treatment.
- This is a freeze/replay and artefact-resolution notebook. It is not Notebook 09 replacement and does not authorise production use.
- Binary `.joblib`, `.npz` and `.parquet` artefacts remain in Google Drive. This notebook validates them in place and exports uploadable JSON/CSV/TXT evidence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow joblib scikit-learn opencv-python-headless


## 1. Configuration and run identity


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, traceback, zipfile, os, re
import numpy as np
import pandas as pd
import joblib

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Set only when deliberately pinning an exact completed Notebook 50 run.
NOTEBOOK50_RUN_OVERRIDE = None

# Candidate roots. The notebook searches recursively because earlier notebooks used both
# direct run folders and a runs/<timestamp> convention.
NOTEBOOK50_ROOTS = [
    ROOT/'observable_residual_mask_m5_v8_7',
    ROOT/'asymmetric_residual_mask_m5_v8_7',
    ROOT/'residual_mask_m5_v8_7',
]

OUTPUT_ROOT = ROOT/'m5_remove_policy_freeze_v8_8a'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook52_v8_8a'
OUT = OUTPUT_ROOT/RUN_ID
AUDIT = OUT/'audit'; TABLES = OUT/'tables'; BUNDLE = OUT/'policy_bundle'; REPLAY = OUT/'replay'; UPLOAD = OUT/'upload'
for p in [OUT,AUDIT,TABLES,BUNDLE,REPLAY,UPLOAD]: p.mkdir(parents=True, exist_ok=False)

EXPECTED_NOTEBOOK50_VERDICT = 'M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER'
EXPECTED_POLICY = 'REMOVE-only'
REQUIRED_CONTROL_FILES = [
    'experiment_identity.json',
    'notebook50_verdict.json',
    'selected_models.json',
    'reason_code_schema.json',
    'inputs/predictor_schema.json',
    'inputs/source_group_split_audit.csv',
    'tables/selected_region_truth_evaluation.csv',
    'tables/m5_map_inventory.csv',
]
OPTIONAL_CONTROL_FILES = [
    'inputs/candidate_fold_manifest.parquet',
    'tables/all_m3_oof_predictions_and_decisions.parquet',
    'tables/asymmetric_target_summary.csv',
    'model_inventory.csv',
    'calibrator_inventory.csv',
    'audit/model_inventory.csv',
    'audit/calibrator_inventory.csv',
    'executive_summary.txt',
]
BINARY_SUFFIXES = {'.joblib','.npz','.parquet','.pt','.pth','.pkl','.pickle','.npy'}
print('Output:', OUT)


## 2. Resolve the authoritative latest Notebook 50 run


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def json_read(path):
    return json.loads(Path(path).read_text())

def required_status(run):
    return {rel:(run/rel).exists() for rel in REQUIRED_CONTROL_FILES}

def candidate_runs():
    rows=[]
    for root in NOTEBOOK50_ROOTS:
        if not root.exists():
            continue
        candidates=[root]+[p for p in root.rglob('*') if p.is_dir()]
        for p in candidates:
            status=required_status(p)
            evidence=sum(status.values())
            if evidence:
                rows.append({'path':str(p),'mtime':p.stat().st_mtime,'required_present':evidence,
                             'required_total':len(REQUIRED_CONTROL_FILES),'complete':all(status.values()),
                             'missing':'|'.join(k for k,v in status.items() if not v)})
    return pd.DataFrame(rows).drop_duplicates('path') if rows else pd.DataFrame(columns=['path','mtime','required_present','required_total','complete','missing'])

if NOTEBOOK50_RUN_OVERRIDE is not None:
    N50=Path(NOTEBOOK50_RUN_OVERRIDE)
    if not N50.exists(): raise RuntimeError(f'Notebook 50 override does not exist: {N50}')
    status=required_status(N50)
    if not all(status.values()): raise RuntimeError(f'Notebook 50 override is incomplete: {[k for k,v in status.items() if not v]}')
    resolution='explicit_override'
    candidates=candidate_runs()
else:
    candidates=candidate_runs()
    candidates.to_csv(AUDIT/'notebook50_run_candidates.csv',index=False)
    if candidates.empty: raise RuntimeError(f'No Notebook 50 candidate directories found below: {NOTEBOOK50_ROOTS}')
    newest=candidates.sort_values('mtime').iloc[-1]
    if not bool(newest.complete):
        raise RuntimeError('The last modified Notebook 50 candidate is incomplete. This notebook will not silently use an older run. '
                           f"Newest candidate: {newest.path}; missing: {newest.missing}")
    N50=Path(newest.path); resolution='latest_complete_and_latest_candidate'

print('Authoritative Notebook 50 run:',N50)
print('Resolution:',resolution)


## 3. Validate control records and the REMOVE-only verdict


In [ ]:
identity=json_read(N50/'experiment_identity.json')
verdict=json_read(N50/'notebook50_verdict.json')
selected_models=json_read(N50/'selected_models.json')
schema=json_read(N50/'inputs'/'predictor_schema.json')
reason_schema=json_read(N50/'reason_code_schema.json')

verdict_text=str(verdict.get('verdict', verdict.get('decision', verdict.get('status',''))))
policy_text=' '.join(str(verdict.get(k,'')) for k in ['m5_policy','policy','selected_policy','decision'])
checks={
    'source_directory_exists':N50.exists(),
    'all_required_control_files_present':all((N50/r).exists() for r in REQUIRED_CONTROL_FILES),
    'remove_only_verdict':verdict_text==EXPECTED_NOTEBOOK50_VERDICT,
    'safe_protect_not_selected':('PROTECT_ONLY' not in verdict_text and 'COMBINED' not in verdict_text),
    'predictor_schema_nonempty':bool(schema.get('ordered_predictors') or schema.get('predictors') or schema.get('ordered_features')),
    'selected_models_record_nonempty':bool(selected_models),
}
if not all(checks.values()):
    (AUDIT/'control_record_failure.json').write_text(json.dumps({'checks':checks,'verdict_text':verdict_text,'policy_text':policy_text},indent=2))
    raise RuntimeError(f'Notebook 50 control-record validation failed: {checks}')
print(json.dumps(checks,indent=2))
print('Verdict:',verdict_text)


## 4. Inventory every durable artefact in place


In [ ]:
def role_for(path):
    s=str(path.relative_to(N50)).lower()
    if '/models/' in '/'+s or path.suffix.lower() in {'.pt','.pth'}: return 'model'
    if 'calibrator' in s or path.suffix.lower()=='.joblib': return 'calibrator_or_serialised_object'
    if path.suffix.lower()=='.npz': return 'semantic_array_container'
    if path.suffix.lower()=='.parquet': return 'tabular_binary'
    if 'schema' in s: return 'schema'
    if 'verdict' in s: return 'verdict'
    if 'manifest' in s or 'identity' in s or 'inventory' in s: return 'manifest_or_identity'
    if path.suffix.lower()=='.csv': return 'review_table'
    if path.suffix.lower()=='.json': return 'json_record'
    return 'other'

rows=[]
for p in sorted(N50.rglob('*')):
    if not p.is_file(): continue
    rel=str(p.relative_to(N50))
    try:
        digest=sha256_file(p); error=None
    except Exception as e:
        digest=None; error=repr(e)
    rows.append({'relative_path':rel,'suffix':p.suffix.lower(),'bytes':p.stat().st_size,'sha256':digest,
                 'role':role_for(p),'binary':p.suffix.lower() in BINARY_SUFFIXES,'hash_error':error})
inventory=pd.DataFrame(rows)
inventory.to_csv(TABLES/'notebook50_full_artifact_inventory.csv',index=False)
if inventory.empty or inventory.sha256.isna().any(): raise RuntimeError('Artifact inventory is empty or contains hash failures.')
print('Inventoried files:',len(inventory),'Binary files:',int(inventory.binary.sum()))
display(inventory.groupby(['role','suffix'],dropna=False).size().reset_index(name='files'))


## 5. Loadability audit for `.joblib`, `.npz` and `.parquet`


In [ ]:
load_rows=[]
for _,r in inventory[inventory.binary].iterrows():
    p=N50/r.relative_path; result={'relative_path':r.relative_path,'role':r.role,'suffix':r.suffix,'load_ok':False,
                                  'object_type':None,'keys_or_columns':None,'shape_or_rows':None,'error':None}
    try:
        if p.suffix.lower()=='.joblib':
            obj=joblib.load(p); result['object_type']=f'{obj.__class__.__module__}.{obj.__class__.__name__}'
            attrs=[x for x in ['predict','predict_proba','transform','classes_','n_features_in_','feature_names_in_'] if hasattr(obj,x)]
            result['keys_or_columns']='|'.join(attrs); result['load_ok']=True
        elif p.suffix.lower()=='.npz':
            with np.load(p,allow_pickle=False) as z:
                keys=list(z.files); result['keys_or_columns']='|'.join(keys)
                result['shape_or_rows']='|'.join(f'{k}:{tuple(z[k].shape)}:{z[k].dtype}' for k in keys[:20])
                if any(not np.isfinite(z[k]).all() for k in keys if np.issubdtype(z[k].dtype,np.number)):
                    raise RuntimeError('Non-finite numeric array present')
            result['object_type']='numpy.lib.npyio.NpzFile'; result['load_ok']=True
        elif p.suffix.lower()=='.parquet':
            df=pd.read_parquet(p); result['object_type']='pandas.DataFrame'; result['keys_or_columns']='|'.join(map(str,df.columns)); result['shape_or_rows']=str(df.shape); result['load_ok']=True
        elif p.suffix.lower()=='.npy':
            a=np.load(p,allow_pickle=False); result['object_type']='numpy.ndarray'; result['shape_or_rows']=f'{a.shape}:{a.dtype}'; result['load_ok']=bool(np.isfinite(a).all()) if np.issubdtype(a.dtype,np.number) else True
        else:
            result['object_type']='hash_only'; result['load_ok']=True
    except Exception as e:
        result['error']=repr(e)
    load_rows.append(result)
load_audit=pd.DataFrame(load_rows)
load_audit.to_csv(TABLES/'binary_artifact_loadability.csv',index=False)
failed=load_audit[~load_audit.load_ok]
print('Binary artefacts audited:',len(load_audit),'Failed:',len(failed))
if len(failed): display(failed)


## 6. Validate page-level M5 decision maps and construct golden semantic hashes


In [ ]:
map_inventory=pd.read_csv(N50/'tables'/'m5_map_inventory.csv')
required_map_cols={'page','file'}
if not required_map_cols.issubset(map_inventory.columns):
    raise RuntimeError(f'm5_map_inventory.csv missing columns: {sorted(required_map_cols-set(map_inventory.columns))}')

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def resolve_saved_path(raw):
    p=Path(str(raw))
    candidates=[p, N50/p, N50/'maps'/p.name, N50/'masks'/p.name]
    hits=[x for x in candidates if x.exists()]
    if len(hits)!=1: raise RuntimeError(f'Expected one durable map for {raw}; found {hits}')
    return hits[0]

map_rows=[]
for _,r in map_inventory.iterrows():
    p=resolve_saved_path(r['file'])
    with np.load(p,allow_pickle=False) as z:
        keys=list(z.files); arrays={k:z[k] for k in keys}
    lower={k.lower():k for k in keys}
    def pick(names):
        for n in names:
            if n.lower() in lower:return arrays[lower[n.lower()]]
        return None
    m3=pick(['M3','m3','m3_mask'])
    sr=pick(['SAFE_REMOVE','safe_remove','remove','remove_decision'])
    sp=pick(['SAFE_PROTECT','safe_protect','protect','protect_decision'])
    m5=pick(['M5','M5_REMOVE','m5','m5_remove'])
    if m3 is None or sr is None: raise RuntimeError(f'{p}: M3 and SAFE_REMOVE arrays are required; keys={keys}')
    m3=np.asarray(m3)>0; sr=np.asarray(sr)>0
    if m3.shape!=sr.shape: raise RuntimeError(f'{p}: M3/SAFE_REMOVE shape mismatch')
    if np.any(sr & ~m3): raise RuntimeError(f'{p}: SAFE_REMOVE is not a subset of M3')
    rebuilt=m3 & ~sr
    if m5 is not None:
        m5=np.asarray(m5)>0
        if m5.shape!=m3.shape or not np.array_equal(m5,rebuilt): raise RuntimeError(f'{p}: saved M5 does not equal M3 minus SAFE_REMOVE')
    map_rows.append({'page':str(r.page),'relative_path':str(p.relative_to(N50)),'keys':'|'.join(keys),'shape':str(m3.shape),
                     'm3_pixels':int(m3.sum()),'safe_remove_pixels':int(sr.sum()),'safe_protect_pixels':int((np.asarray(sp)>0).sum()) if sp is not None else None,
                     'm5_remove_pixels':int(rebuilt.sum()),'safe_remove_subset_m3':True,'saved_m5_exact':True if m5 is not None else None,
                     'm3_semantic_sha256':array_hash(m3.astype(np.uint8)),'safe_remove_semantic_sha256':array_hash(sr.astype(np.uint8)),
                     'm5_remove_semantic_sha256':array_hash(rebuilt.astype(np.uint8))})
map_audit=pd.DataFrame(map_rows).sort_values('page')
map_audit.to_csv(TABLES/'m5_remove_golden_master_semantic_hashes.csv',index=False)
print('Validated page maps:',len(map_audit),'SAFE_REMOVE pixels:',int(map_audit.safe_remove_pixels.sum()))


## 7. Replay saved tabular decisions against map totals where possible


In [ ]:
replay_checks={'golden_maps_validated':len(map_audit)==len(map_inventory),'all_binary_loadable':bool(len(load_audit)==0 or load_audit.load_ok.all())}
parquet_candidates=[N50/'tables'/'all_m3_oof_predictions_and_decisions.parquet']
parquet_candidates += [N50/p for p in inventory.loc[inventory.suffix=='.parquet','relative_path']]
parquet_candidates=list(dict.fromkeys(p for p in parquet_candidates if p.exists()))
replay_detail=[]
for p in parquet_candidates:
    df=pd.read_parquet(p)
    cols={c.lower():c for c in df.columns}
    page_col=next((cols[x] for x in ['page','page_id'] if x in cols),None)
    decision_col=next((cols[x] for x in ['safe_remove','remove_decision','decision_remove','is_safe_remove'] if x in cols),None)
    if page_col and decision_col:
        q=df.copy(); decisions=q[decision_col].astype(bool)
        observed=q.loc[decisions].groupby(page_col).size().to_dict()
        for _,r in map_audit.iterrows():
            replay_detail.append({'source':str(p.relative_to(N50)),'page':r.page,'table_safe_remove_rows':int(observed.get(r.page,0)),
                                  'map_safe_remove_pixels':int(r.safe_remove_pixels),'exact_count_match':int(observed.get(r.page,0))==int(r.safe_remove_pixels)})
replay_df=pd.DataFrame(replay_detail)
replay_df.to_csv(TABLES/'tabular_to_map_replay.csv',index=False)
replay_checks['tabular_replay_available']=bool(len(replay_df))
replay_checks['tabular_map_counts_exact']=bool(len(replay_df) and replay_df.exact_count_match.all()) if len(replay_df) else None
print(json.dumps(replay_checks,indent=2))


## 8. Freeze the V8.8A policy record


In [ ]:
predictors=schema.get('ordered_predictors',schema.get('predictors',schema.get('ordered_features',[])))
# Copy small, uploadable control records. Binary learned objects remain in the authoritative source run.
copy_records=[r for r in REQUIRED_CONTROL_FILES if (N50/r).suffix.lower() not in BINARY_SUFFIXES]
for rel in copy_records:
    src=N50/rel; dst=BUNDLE/Path(rel).name
    shutil.copy2(src,dst)

bundle_manifest={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.8A-notebook52-m5-remove-freeze-and-replay',
    'development_engineering_only':True,
    'promotion_authority':False,
    'authoritative_notebook50_run':str(N50),
    'source_resolution':resolution,
    'source_experiment_identity_sha256':sha256_file(N50/'experiment_identity.json'),
    'source_verdict_sha256':sha256_file(N50/'notebook50_verdict.json'),
    'source_verdict':verdict_text,
    'policy':'M5_REMOVE',
    'policy_definition':'M5_REMOVE = M3 minus SAFE_REMOVE',
    'safe_protect_runtime_enabled':False,
    'm3_fallback':True,
    'retraining_performed':False,
    'recalibration_performed':False,
    'threshold_changed':False,
    'ordered_predictors':predictors,
    'predictor_schema_sha256':sha256_file(N50/'inputs'/'predictor_schema.json'),
    'page_maps':len(map_audit),
    'safe_remove_pixels':int(map_audit.safe_remove_pixels.sum()),
    'golden_master_csv_sha256':sha256_file(TABLES/'m5_remove_golden_master_semantic_hashes.csv'),
    'full_artifact_inventory_sha256':sha256_file(TABLES/'notebook50_full_artifact_inventory.csv'),
    'binary_loadability_sha256':sha256_file(TABLES/'binary_artifact_loadability.csv'),
    'control_checks':checks,
    'replay_checks':replay_checks,
    'binary_objects_remain_in_place':True,
}
bundle_manifest['policy_id']=hashlib.sha256(json.dumps({k:bundle_manifest[k] for k in ['source_experiment_identity_sha256','source_verdict_sha256','policy','predictor_schema_sha256','golden_master_csv_sha256']},sort_keys=True).encode()).hexdigest()
(BUNDLE/'policy_manifest.json').write_text(json.dumps(bundle_manifest,indent=2))
print('Policy ID:',bundle_manifest['policy_id'])


## 9. Fail-closed V8.8A verdict


In [ ]:
gate_order=['source_complete','remove_only_verdict','binary_loadability','map_integrity','golden_master_replay']
gates={
 'source_complete':bool(checks['all_required_control_files_present']),
 'remove_only_verdict':bool(checks['remove_only_verdict'] and checks['safe_protect_not_selected']),
 'binary_loadability':bool(len(load_audit)>0 and load_audit.load_ok.all()),
 'map_integrity':bool(len(map_audit)==len(map_inventory) and map_audit.safe_remove_subset_m3.all() and map_audit.saved_m5_exact.dropna().all()),
 'golden_master_replay':bool(replay_checks['golden_maps_validated']),
}
first_failed=next((g for g in gate_order if not gates[g]),None)
verdict52='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER' if first_failed is None else 'FAIL_M5_REMOVE_FREEZE'
verdict_record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict52,'first_failed_gate':first_failed,'gates':gates,
                'policy_id':bundle_manifest['policy_id'],'authoritative_notebook50_run':str(N50),
                'meaning':'PASS authorises Notebook 53 construction against this frozen source identity; it does not replace Notebook 09 or authorise production.'}
(OUT/'notebook52_verdict.json').write_text(json.dumps(verdict_record,indent=2))
pd.DataFrame([{'gate':g,'passed':gates[g]} for g in gate_order]).to_csv(TABLES/'notebook52_gate_results.csv',index=False)
print(json.dumps(verdict_record,indent=2))
if first_failed is not None: raise RuntimeError(f'Notebook 52 failed closed at {first_failed}. See {OUT}')


## 10. Exact second-run replay candidate

Run the notebook a second time unchanged. The source fingerprint below must be identical. Because each execution has a new output directory, the source fingerprint is compared with the prior successful Notebook 52 run if one exists.


In [ ]:
fingerprint_payload={
 'authoritative_notebook50_run':str(N50),
 'source_experiment_identity_sha256':bundle_manifest['source_experiment_identity_sha256'],
 'source_verdict_sha256':bundle_manifest['source_verdict_sha256'],
 'predictor_schema_sha256':bundle_manifest['predictor_schema_sha256'],
 'golden_master_csv_sha256':bundle_manifest['golden_master_csv_sha256'],
 'policy_id':bundle_manifest['policy_id'],
}
source_fingerprint=hashlib.sha256(json.dumps(fingerprint_payload,sort_keys=True).encode()).hexdigest()
prior=[]
for p in OUTPUT_ROOT.iterdir() if OUTPUT_ROOT.exists() else []:
    q=p/'replay'/'source_fingerprint.json'
    if p!=OUT and q.exists():
        try: prior.append((p.stat().st_mtime,p,json_read(q)))
        except Exception: pass
prior=sorted(prior)
prior_match=None if not prior else prior[-1][2].get('source_fingerprint')==source_fingerprint
replay_record={'created_utc':datetime.now(timezone.utc).isoformat(),'source_fingerprint':source_fingerprint,'payload':fingerprint_payload,
               'prior_notebook52_run':str(prior[-1][1]) if prior else None,'exact_match_to_prior':prior_match,
               'interpretation':'None means first candidate. True means unchanged second-run source identity. False is a stop condition before Notebook 53.'}
(REPLAY/'source_fingerprint.json').write_text(json.dumps(replay_record,indent=2))
print(json.dumps(replay_record,indent=2))


## 11. Final upload package


In [ ]:
summary_lines=[
 'NOTEBOOK 52 / V8.8A M5_REMOVE FREEZE AND REPLAY','',
 f'Verdict: {verdict52}',f'Policy ID: {bundle_manifest["policy_id"]}',
 f'Authoritative Notebook 50 run: {N50}',f'Source resolution: {resolution}',
 f'Notebook 50 verdict: {verdict_text}',f'Page maps validated: {len(map_audit)}',
 f'SAFE_REMOVE pixels: {int(map_audit.safe_remove_pixels.sum())}',
 f'Binary artefacts audited: {len(load_audit)}',f'Binary load failures: {len(failed)}',
 f'Source fingerprint: {source_fingerprint}',f'Exact match to prior Notebook 52 run: {prior_match}','',
 'SAFE_PROTECT runtime enabled: False','Retrained: False','Recalibrated: False','Threshold changed: False','',
 'A PASS authorises construction of Notebook 53 against this frozen source identity only. It does not replace Notebook 09 or authorise production.'
]
(OUT/'executive_summary.txt').write_text(chr(10).join(summary_lines))

upload_files=[
 OUT/'executive_summary.txt', OUT/'notebook52_verdict.json', BUNDLE/'policy_manifest.json', REPLAY/'source_fingerprint.json',
 TABLES/'notebook50_full_artifact_inventory.csv', TABLES/'binary_artifact_loadability.csv',
 TABLES/'m5_remove_golden_master_semantic_hashes.csv', TABLES/'tabular_to_map_replay.csv',
 TABLES/'notebook52_gate_results.csv', AUDIT/'notebook50_run_candidates.csv'
]
upload_rows=[]
for src in upload_files:
    if not src.exists(): continue
    dst=UPLOAD/src.name; shutil.copy2(src,dst)
    upload_rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst),'source':str(src.relative_to(OUT))})
pd.DataFrame(upload_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
readme='''Notebook 52 V8.8A upload package

This curated package contains uploadable evidence only. Binary model, calibrator, NPZ and parquet objects remain in the authoritative Notebook 50 Google Drive run and are identified in notebook50_full_artifact_inventory.csv.
'''
(UPLOAD/'README.txt').write_text(readme)
zip_path=UPLOAD/'notebook52_v8_8a_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path: z.write(p,p.name)
if not zip_path.exists() or zip_path.stat().st_size==0: raise RuntimeError('Upload bundle creation failed.')
print(chr(10).join(summary_lines)); print('Upload bundle:',zip_path)


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook52_verdict.json`
3. `policy_bundle/policy_manifest.json`
4. `tables/binary_artifact_loadability.csv`
5. `tables/m5_remove_golden_master_semantic_hashes.csv`
6. `replay/source_fingerprint.json`
7. `upload/notebook52_v8_8a_upload_bundle.zip`

If the notebook fails, do not suppress the error or select an older Notebook 50 run merely to obtain a pass. Correct the first failed source, integrity, loadability or map-replay gate.
